# P3 방사청 국외조달 정제 → `clean_` 3표 + 표기 통일 사전 적재

작성 2026-09-19 (담당분배 명세 `docs/reference/clean-conversion-spec-2026-09-18.md` §1·§4). 규칙 근거는 `docs/reference/data-cleaning-rules.md` §1·§2-6·§2-8·§3, DDL은 `db/alter_2026-09-19_p3_clean.sql`(적용 2026-09-19, admin).

| 원본 파일(데이터셋 키) | 정제 결과(clean_) | 기대 행 수 | 비고 |
|---|---|---|---|
| `raw_dapa_overseas_plan_api` 13,615 | `clean_dapa_overseas_plan_api` — 조달요구번호+품목순번 | 13,615 (제외 0) | 금액은 통화 미검증 → `amount_unverified=1`, 합산 금지 |
| (위 표의 적용장비명) | `ref_equipment_alias` — 원문 → 표준명(잠정) | 원문 843종 | 자동 매핑은 `link_status='후보'`, 근거 없으면 `name_std` NULL |
| `raw_dapa_overseas_contract` 6,333 | `clean_dapa_overseas_contract` — 계약번호 | 6,327 (제외 6: 테스트 계약 `PLACEHOLDER`) | 담당자명 제외, 업체명으로 국가 추정 금지. 업체명 `TEST2` 6행 제외(P3 검수 09-20) |
| `raw_dapa_overseas_bid_result` 2,494 | `clean_dapa_overseas_bid_result` — PK `raw_row_id` | 2,494 (제외 0) | 개찰 2025-03~09 부분연도 |
| `ref_fsg`·`ref_fsc` | 검수(FSG 60 전자 플래그 0→1) | — | alter 에서 적용, 여기서는 확인만 |

`clean_dapa_overseas_plan`(파일판, 3,023행)은 이미 적재돼 있어 건드리지 않는다.

원칙
- 원본은 `data/raw/` 파일을 `read_raw()`로 읽는다(RDS `raw_` 표는 2026-09-22 삭제 — `raw_row_id` = 파서 순번, 파일명 정렬 × 행 순). 대상 `clean_`이 **비어 있을 때만** 적재한다(재적재는 `TRUNCATE` 후 재실행). 제외한 raw 행은 `clean_excluded_row`에 사유와 함께 남겨 `raw = clean + excluded`를 검산한다(P3는 계약정보 테스트 6행만 제외 기대).
- **판단 속성은 만들지 않는다**: 표기 통일 자동 매핑은 `link_status='후보'`, 변이 근거가 없는 원문은 `name_std=NULL` + `'미확인'`. 소요군 부대명이 국방부 직할로 확인되지 않으면 `army_std='미확인'`.
- **HS6 ↔ FSC 대응은 만들지 않는다**(`ref_category_map` 미확정). 파일판(`clean_dapa_overseas_plan`)과 API판은 판단번호 체계가 달라 조인·합산하지 않는다.
- 금액: API판 `budgetAmount`·`untpc`는 통화 미검증 → 숫자형으로 담되 `amount_unverified=1`, 집계 금지. 입찰결과 `예산금액(달러)`는 A7 원화 예산과 합산하지 않는다.


In [1]:
import sys, re, unicodedata
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from load_db import connect, log_stage, current_user, read_raw, log_raw_stage   # .env 읽기 · meta_load_log 기록 · 원본 파일 읽기
RAW = {}   # 원본 파일 DataFrame 캐시 {데이터셋 키: read_raw 결과}. RDS raw_ 표는 2026-09-22 삭제 — row_id = 파서 순번 = clean.raw_row_id
def raw(table, cols=None):
    if table not in RAW:
        RAW[table] = read_raw(table)
    df = RAW[table]
    return (df[["row_id"] + [c for c in cols if c != "row_id"]] if cols else df).copy()

NB_TAG = "notebooks/clean_p3_overseas.ipynb"
conn = connect()
cur = conn.cursor()
USER = current_user(cur)
NOW = datetime.now().replace(microsecond=0)
def q(sql, params=None):
    cur.execute(sql, params)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
def count(t):
    cur.execute(f"SELECT COUNT(*) FROM {t}"); return cur.fetchone()[0]
def py(v):
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if v is pd.NaT or isinstance(v, pd._libs.missing.NAType): return None
    if hasattr(v, "item") and not isinstance(v, str): return v.item()
    if isinstance(v, pd.Timestamp): return v.to_pydatetime()
    return v
def insert_df(table, df, cols, chunk=2000):
    rows = [tuple(py(v) for v in r) for r in df[cols].itertuples(index=False, name=None)]
    sql = f"INSERT INTO {table} ({','.join(cols)}) VALUES ({','.join(['%s']*len(cols))})"
    for i in range(0, len(rows), chunk):
        cur.executemany(sql, rows[i:i+chunk])
    return len(rows)

DATE_RE = r"^\d{4}-\d{2}-\d{2}$"
def to_date(s):
    return pd.to_datetime(s.where(s.astype(str).str.fullmatch(DATE_RE), None), errors="coerce").dt.date
def to_int(s):
    return pd.to_numeric(s.replace({"": None}), errors="coerce").round().astype("Int64")
def to_dec(s):
    return pd.to_numeric(s.replace({"": None}), errors="coerce")
excluded = []   # (table_name, raw_row_id, reason_code, note)
print("connected as", USER, "| pandas", pd.__version__)

connected as etl_rw@% | pandas 3.0.5


## 1. 국외 조달계획 OpenAPI 품목 단위 → `clean_dapa_overseas_plan_api`

- 키 = `procure_demand_no` × `item_seq`. raw 13,615행에서 조합 고유 13,615(2026-09-19 실측) → PK. 품목순번 공란 842행은 **빈 문자열**로 두고 `is_item_seq_missing=1`(조달요구번호가 서로 달라 키 충돌 없음).
- **NSN 판별**(명세 §4-1): `stock_no` 13자 중 `^[0-9]{13}$` = `숫자13`, NCB 37 영숫자(`^[0-9]{4}[0-9A-Z]{9}$`) = `영숫자13` → 이 둘만 `nsn`에 넣는다. `NSN`·`NSN001`·`N/A`·`TBD`·`0` 반복 같은 자리표시는 `nsn=NULL` + `is_placeholder_nsn=1`, 나머지는 `기타`.
- `fsc4`/`fsg2` = `nsn` 앞 4·2자리, `is_elec = fsg2 IN ('58','59','60')`(명세 §4-2). **v3 문서의 전자·통신 2,270건은 길이와 무관하게 `stock_no` 앞 2자리로 센 값**이라 13자 기준으로는 2,267이 되고 3행 차이가 난다 — 아래 셀에서 둘 다 출력한다.
- `army_std`(명세 §4-4): 육군/해군/공군은 그대로, `해병` → `해병대`, 국방부 직할로 이름에서 확인되는 부대(국방부직할부대·국방부근무지원단·정보사·국군화생방방호사령부·드론작전사령부)는 `국직`, 확인 불가(`3707부대`)는 `미확인`. **공군이 717건(5.3%)뿐**이라 군별 비교는 치우침을 병기해야 한다 → `meta_load_log`에 남긴다.
- 금액(명세 §4-5): `budget_amount`·`unit_price`는 숫자형으로 담되 `amount_unverified=1` 고정. **합산·비교하지 않는다.** `quantity`는 정수.
- KDSIS 연결(명세 §4-7·§7): `nsn` exact → `clean_kdsis_nsn.nsn`. 양쪽 모두 하이픈 없는 13자라 형식은 이미 같다. 연결률·미연결 수를 아래에 출력한다.
- **제외 열**(`data-cleaning-rules.md` §2-6 등급 ✕): `org_name`·`org_code`·`purchase_request_no`·`qa_grade`·`standard_no`·`component_no`는 내부 행정 코드라 clean 에 두지 않는다(필요하면 `raw_row_id`로 raw 참조).


In [2]:
ra = raw("raw_dapa_overseas_plan_api", ["procure_demand_no", "item_seq", "demand_year_req", "army_name", "army_code", "function_name", "function_code", "item_kind_name", "item_kind_code", "item_name", "stock_no", "equipment_code", "equipment_name", "quantity", "unit", "budget_amount", "unit_price", "progress_status"])
print("원본 전체:", len(ra))
a = ra.copy()
a["item_seq"] = a["item_seq"].fillna("").astype(str)
a["is_item_seq_missing"] = (a["item_seq"] == "").astype(int)
key = ["procure_demand_no", "item_seq"]
print("키 고유:", a[key].drop_duplicates().shape[0], "| 품목순번 공란:", int(a["is_item_seq_missing"].sum()))
assert a[key].drop_duplicates().shape[0] == len(a), "키 중복 — PK 설계 재검토 필요"

# NSN 판별
s = a["stock_no"].fillna("").astype(str).str.strip()
a["stock_no_raw"] = s
num13 = s.str.fullmatch(r"[0-9]{13}")
aln13 = s.str.fullmatch(r"[0-9]{4}[0-9A-Z]{9}") & ~num13
PLACEHOLDER = r"(?i)(nsn[\s\-\w/\.]*|n\s*/?\s*a(n\s*/?\s*a)*|tbd(tbd)*|[\-\*\.]+|0+)"
ph = ~(num13 | aln13) & s.str.fullmatch(PLACEHOLDER)
a["nsn_format"] = np.select([num13, aln13, ph], ["숫자13", "영숫자13", "자리표시"], default="기타")
a["is_placeholder_nsn"] = ph.astype(int)
a["nsn"] = s.where(num13 | aln13)
a["fsc4"] = a["nsn"].str[:4]
a["fsg2"] = a["nsn"].str[:2]
a["is_elec"] = a["fsg2"].isin(["58", "59", "60"]).astype(int)
print("nsn_format:", a["nsn_format"].value_counts().to_dict())
print("is_elec(13자 nsn 기준):", int(a["is_elec"].sum()),
      "| 참고 v3 방식(길이 무관 stock_no 앞2자리):", int(s.str[:2].isin(["58", "59", "60"]).sum()))
print("fsg2 내역:", a.loc[a["is_elec"] == 1, "fsg2"].value_counts().to_dict(), "| fsc4 고유:", a["fsc4"].nunique())

# KDSIS 연결 (형식: 양쪽 모두 하이픈 없는 13자)
kd = set(q("SELECT nsn FROM clean_kdsis_nsn")["nsn"])
a["kdsis_link_status"] = np.where(a["nsn"].isna(), "대상아님", np.where(a["nsn"].isin(kd), "연결", "미연결"))
print("kdsis_link_status:", a["kdsis_link_status"].value_counts().to_dict())
print("NSN 고유:", a["nsn"].nunique(), "| 그중 KDSIS 연결 고유:", a.loc[a["kdsis_link_status"] == "연결", "nsn"].nunique())
print("다중 일치: 없음(clean_kdsis_nsn.nsn 은 PK — 1:1 또는 0)")

원본 전체: 13615
키 고유: 13615 | 품목순번 공란: 842
nsn_format: {'숫자13': 9970, '영숫자13': 3266, '자리표시': 347, '기타': 32}
is_elec(13자 nsn 기준): 2267 | 참고 v3 방식(길이 무관 stock_no 앞2자리): 2270
fsg2 내역: {'59': 1795, '58': 461, '60': 11} | fsc4 고유: 325


kdsis_link_status: {'미연결': 12620, '연결': 616, '대상아님': 379}
NSN 고유: 8210 | 그중 KDSIS 연결 고유: 412
다중 일치: 없음(clean_kdsis_nsn.nsn 은 PK — 1:1 또는 0)


In [3]:
# 군 표준값 · 금액 · 진행상태
GUKJIK = {"국방부직할부대", "국방부근무지원단", "정보사", "국군화생방방호사령부", "드론작전사령부"}
def army_std(v):
    if v in ("육군", "해군", "공군"): return v
    if v == "해병": return "해병대"
    if v in GUKJIK: return "국직"
    return "미확인"
a["army_name_raw"] = a["army_name"]
a["army_std"] = a["army_name"].map(army_std).fillna("미확인")
a["quantity"] = to_int(a["quantity"])
a["budget_amount_num"] = to_dec(a["budget_amount"])
a["unit_price_num"] = to_dec(a["unit_price"])
a["amount_unverified"] = 1
a["is_contracted"] = a["progress_status"].isin(["계약", "부분계약"]).astype(int)
a["demand_year"] = pd.to_numeric(a["demand_year_req"], errors="coerce").astype("Int64")
a["raw_row_id"] = a["row_id"]; a["cleaned_at"] = NOW; a["cleaned_by"] = USER
army_cnt = a["army_std"].value_counts().to_dict()
print("army_std:", army_cnt, "| 공군 비중 %:", round(100 * army_cnt.get("공군", 0) / len(a), 1))
print("원문→미확인으로 남긴 값:", sorted(set(a.loc[a["army_std"] == "미확인", "army_name_raw"].dropna())))
print("demand_year:", a["demand_year"].value_counts().sort_index().to_dict())
print("is_contracted:", int(a["is_contracted"].sum()), "| quantity NULL:", int(a["quantity"].isna().sum()),
      "| budget NULL:", int(a["budget_amount_num"].isna().sum()), "| unit_price NULL:", int(a["unit_price_num"].isna().sum()))
print("금액은 amount_unverified=1 — 합계를 내지 않는다(통화 미검증)")

army_std: {'육군': 7404, '해군': 5254, '공군': 717, '해병대': 212, '국직': 18, '미확인': 10} | 공군 비중 %: 5.3
원문→미확인으로 남긴 값: ['3707부대']
demand_year: {np.int64(2016): 2791, np.int64(2017): 3044, np.int64(2018): 1, np.int64(2020): 11, np.int64(2021): 2494, np.int64(2022): 1651, np.int64(2023): 924, np.int64(2024): 892, np.int64(2025): 854, np.int64(2026): 953}
is_contracted: 8478 | quantity NULL: 0 | budget NULL: 0 | unit_price NULL: 0
금액은 amount_unverified=1 — 합계를 내지 않는다(통화 미검증)


## 2. 적용장비명 표기 통일 사전 `ref_equipment_alias` (명세 §4-3)

- 한 행 = **원문 1종**(`equipment_name`). `*`·공란은 사전에 넣지 않고 `clean_dapa_overseas_plan_api.is_equipment_missing=1`로만 표시(원본 2,018행).
- `name_norm` = **기계적 정규화**(NFKC 전각→반각, 대괄호 `[]`→소괄호 `()`, 공백 전부 제거, ASCII 대문자). 판단이 들어가지 않는다.
- `variant_key` = `name_norm` + **표기 변이 규칙**(현재 `레이다`→`레이더` 1건. 외래어 표기 변이 — **잠정**).
- `name_std`는 **같은 `variant_key`에 원문이 2종 이상 모인 묶음**(= 표기 변이가 자료 안에서 실제로 관측된 경우)에만 최빈 원문으로 채우고 `link_status='후보'`. 그 외는 `name_std=NULL` + `'미확인'` — 근거 없는 표준명을 만들지 않는다.
- 장비코드로는 묶지 않는다(명세 §4-3). 대신 원문별 `code_count`를 남겨 코드가 갈라지는 정도를 본다.
- 명세가 먼저 통일하라고 한 범위는 전자·통신 건에 나오는 원문(`in_elec_scope=1`)이다. 사전 자체는 843종 전부를 담되 이 플래그로 우선순위를 표시한다.


In [4]:
def norm_eq(v):
    t = unicodedata.normalize("NFKC", str(v)).strip().upper()
    t = t.replace("[", "(").replace("]", ")")
    return re.sub(r"\s+", "", t)
VARIANT_RULES = [("레이다", "레이더")]
def to_variant_key(t):
    for src, dst in VARIANT_RULES:
        t = t.replace(src, dst)
    return t

eq = a["equipment_name"].fillna("").astype(str).str.strip()
a["equipment_name"] = eq.where(~eq.isin(["", "*"]))
a["is_equipment_missing"] = eq.isin(["", "*"]).astype(int)
a["equipment_name_norm"] = a["equipment_name"].map(norm_eq, na_action="ignore")
print("적용장비명 공란·*:", int(a["is_equipment_missing"].sum()), "| 원문 고유:", a["equipment_name"].nunique())

base = a.loc[a["equipment_name"].notna(), ["equipment_name", "equipment_code", "is_elec", "row_id"]]
al = base.groupby("equipment_name").agg(row_count=("row_id", "size"),
                                        elec_row_count=("is_elec", "sum"),
                                        code_count=("equipment_code", "nunique")).reset_index()
al = al.rename(columns={"equipment_name": "name_raw"})
al["name_norm"] = al["name_raw"].map(norm_eq)
al["variant_key"] = al["name_norm"].map(to_variant_key)
al["in_elec_scope"] = (al["elec_row_count"] > 0).astype(int)
al["variant_group_size"] = al.groupby("variant_key")["name_raw"].transform("size").astype(int)
# 묶음 대표(최빈 원문, 동수면 짧은 쪽 → 사전순). 잠정 규칙.
al = al.sort_values(["variant_key", "row_count", "name_raw"], ascending=[True, False, True])
rep = al.drop_duplicates("variant_key").set_index("variant_key")["name_raw"]
multi = al["variant_group_size"] > 1
al["name_std"] = np.where(multi, al["variant_key"].map(rep), None)
al["link_status"] = np.where(multi, "후보", "미확인")
al["basis"] = np.where(multi,
    "자동 매핑(잠정): NFKC·대괄호→소괄호·공백 제거·대문자 + 레이다→레이더 후 최빈 원문 채택",
    "표기 변이가 자료에 없음 — 표준명 미정(근거 없는 표준명을 만들지 않음)")
al["decided_at"] = None
print("사전 행(원문 종수):", len(al), "| 전자 범위:", int(al["in_elec_scope"].sum()),
      "| link_status:", al["link_status"].value_counts().to_dict())
print("표기 변이 묶음(원문 2종 이상):", int(al.loc[multi, "variant_key"].nunique()), "묶음,", int(multi.sum()), "원문")
print("장비코드 2개 이상 붙은 원문:", int((al["code_count"] > 1).sum()), "| 최대 코드 수:", int(al["code_count"].max()))
ex = al[multi].sort_values(["variant_group_size", "row_count"], ascending=False).head(8)
for r in ex.itertuples():
    print(f"  예시 {r.variant_key[:28]!r} ← {r.name_raw!r} (std={r.name_std!r}, 묶음 {r.variant_group_size})")
std_map = dict(al[["name_raw", "name_std"]].itertuples(index=False, name=None))
a["equipment_name_std"] = a["equipment_name"].map(std_map)
print("표준명이 채워진 행:", int(a["equipment_name_std"].notna().sum()), "/", int(a["equipment_name"].notna().sum()))

적용장비명 공란·*: 2018 | 원문 고유: 843


사전 행(원문 종수): 843 | 전자 범위: 365 | link_status: {'미확인': 803, '후보': 40}
표기 변이 묶음(원문 2종 이상): 20 묶음, 40 원문
장비코드 2개 이상 붙은 원문: 60 | 최대 코드 수: 11
  예시 '레이더세트' ← '레이더 세트' (std='레이더 세트', 묶음 2)
  예시 '레이더세트(대포병탐지레이더)' ← '레이더 세트(대포병탐지레이더)' (std='레이더 세트(대포병탐지레이더)', 묶음 2)
  예시 '헬기유도등장치' ← '헬기유도등장치' (std='헬기유도등장치', 묶음 2)
  예시 '엔진,디젤식' ← '엔진,디젤식' (std='엔진,디젤식', 묶음 2)
  예시 '레이더세트' ← '레이더세트' (std='레이더 세트', 묶음 2)
  예시 '레이더세트,PKG' ← '레이더 세트, PKG' (std='레이더 세트, PKG', 묶음 2)
  예시 '헬기유도등장치' ← '헬기 유도등 장치' (std='헬기유도등장치', 묶음 2)
  예시 '레이더세트(대포병탐지레이더)' ← '레이더 세트(대포병탐지레이다)' (std='레이더 세트(대포병탐지레이더)', 묶음 2)
표준명이 채워진 행: 439 / 11597


## 3. 국외조달 계약정보 → `clean_dapa_overseas_contract`

- 키 = `contract_no`(raw 고유 6,333 = 전체행, 2026-09-19 실측) → PK. 중복·열 밀림 없음.
- `contract_date`는 전 행 `YYYY-MM-DD`(2017-02-01~2025-12-31). `contract_period`는 `YYYY-MM-DD~YYYY-MM-DD` 5,602 + `YYYY-MM-DD~`(종료일 없음) 731 = 6,333 → `period_start`/`period_end` + `is_open_ended`.
- **담당자명**(`contract_org_officer_name`)은 읽지 않는다. 단일값 4열(`계약기관구분명`·`계약기관명`·`수요기관구분명`·`수요기관명` = 전부 `국가기관`/`방위사업청`, `업무구분명` = 전부 `외자`)은 정보량이 없어 clean 에 두지 않는다.
- **테스트 계약 제외**(P3 검수 09-20 회신, `db/alter_2026-09-20_p3_test_vendor.sql`): 대표업체명이 정확히 `TEST2`인 6행(raw_row_id 202·203·709·713·1558·1559 — 계약명 `한도액계약테스트`·`인지세TEST` 등, 계약번호 `TEST…`·`TE00…`)은 `clean_excluded_row` `PLACEHOLDER`로 제외한다. 키워드 `TEST`/`테스트`로는 거르지 않는다(`REPAIR FOR AGM-65 TEST SYSTEMS`·`시험세트(TEST SET, RADAR)` 등 실제 계약 19행). P2→P4 테스트 업체 6행 제외(09-19)와 같은 처리.
- `vendor_name`은 외국 업체명이며 **국가를 추정하지 않는다**. 금액·국가 열이 원본에 없어 집중도·규모 계산은 불가(`data-cleaning-rules.md` §2-8).


In [5]:
rc = raw("raw_dapa_overseas_contract", ["contract_no", "contract_name", "contract_form_name", "contract_method_name", "contract_date", "contract_period", "contract_org_dept_name", "vendor_name"])
print("원본 전체:", len(rc), "| 계약번호 고유:", rc["contract_no"].nunique())
assert rc["contract_no"].is_unique
test_mask = rc["vendor_name"].eq("TEST2")   # 테스트 계약 6행 — 정확히 이 값만(키워드 필터 아님)
for r in rc[test_mask].itertuples(index=False):
    excluded.append(("raw_dapa_overseas_contract", int(r.row_id), "PLACEHOLDER",
                     f"테스트 계약: 업체명 TEST2 ({r.contract_no}, {r.contract_name}). P3 검수 09-20, 제외 결정 09-20"))
print("테스트 계약 제외:", int(test_mask.sum()), "행 (기대 6) →", sorted(rc.loc[test_mask, "row_id"].tolist()))
assert int(test_mask.sum()) == 6
c = rc[~test_mask].copy()
c["contract_date"] = to_date(c["contract_date"]); assert c["contract_date"].notna().all()
c["contract_year"] = pd.to_datetime(c["contract_date"]).dt.year
per = c["contract_period"].fillna("").astype(str).str.split("~", n=1, expand=True)
c["period_start"] = to_date(per[0]); c["period_end"] = to_date(per[1].fillna(""))
c["is_open_ended"] = ((c["period_start"].notna()) & (c["period_end"].isna())).astype(int)
c["raw_row_id"] = c["row_id"]; c["cleaned_at"] = NOW; c["cleaned_by"] = USER
print("연도별 건수:", c["contract_year"].value_counts().sort_index().to_dict())
print("period_start NULL:", int(c["period_start"].isna().sum()), "| 종료일 없음(is_open_ended):", int(c["is_open_ended"].sum()))
print("고유 업체 수(대표업체명 원문):", c["vendor_name"].nunique(), "— 국가 추정 금지")
print("계약체결형태:", c["contract_form_name"].value_counts().to_dict())

원본 전체: 6333 | 계약번호 고유: 6333
테스트 계약 제외: 6 행 (기대 6) → [202, 203, 709, 713, 1558, 1559]
연도별 건수: {2017: 703, 2018: 850, 2019: 824, 2020: 813, 2021: 801, 2022: 667, 2023: 499, 2024: 575, 2025: 595}
period_start NULL: 0 | 종료일 없음(is_open_ended): 725
고유 업체 수(대표업체명 원문): 444 — 국가 추정 금지
계약체결형태: {'총액제': 2832, '단가제(최저가)': 2173, '내역입찰(최저가)': 1153, '리스입찰': 162, '단가제': 4, '내역입찰(적격심사)': 3}


## 4. 국외조달 입찰결과 → `clean_dapa_overseas_bid_result`

- **업무 식별자 단독으로는 고유하지 않다**(2026-09-19 실측: 공고번호 고유 14 · 판단번호 97 · 판단번호+항목번호 1,362 · 공고번호+항목번호 467). 조합 (`공고번호`,`판단번호`,`항목번호`,`개찰일시`)만 2,494로 고유 → **PK는 `raw_row_id`**, 그 조합은 UNIQUE 로만 둔다.
- `bid_notice_no`는 `EHG0001-1` 형태(공고번호-차수) → `notice_no_base`/`notice_seq`로 분리.
- 개찰일시 **2025-03-27 ~ 2025-09-15** = 부분연도 → `is_partial_year=1` 전 행. 연간 유찰률로 표현하지 않는다.
- `bid_result` 표준화: 원본 값은 `유찰`/`낙찰` 2종뿐(각각 2,146 / 348) → `bid_result_std`. 그 외 값이 나오면 `미확인`.
  `data-cleaning-rules.md` §2-8은 "낙찰 342·유찰 1,020·재공고 1,126"으로 적혀 있으나 **현재 raw 실측과 다르다** — 아래 출력값으로 문서를 고친다.
- `budget_amount_usd`는 숫자형(결측 0). **A7 원화 예산과 합산하지 않는다.**
- `plan_link_status`: `decision_no`가 파일판 `raw_dapa_overseas_plan`에 있으면 `연결`(판단번호 97 중 83 교집합). 행 단위 조인은 하지 않는다.
- 단일값 2열(`단가제유형`·`사전심사` = 전부 `해당없음`)은 clean 에 두지 않는다.


In [6]:
rb = raw("raw_dapa_overseas_bid_result", ["bid_notice_no", "decision_no", "item_seq", "bid_item_name", "ordering_agency", "contract_method", "bid_method", "award_method", "opening_datetime", "bid_result", "budget_amount_usd"])
print("원본 전체:", len(rb))
b = rb.copy()
kcols = ["bid_notice_no", "decision_no", "item_seq", "opening_datetime"]
print("고유성 — 공고번호:", b["bid_notice_no"].nunique(), "| 판단번호:", b["decision_no"].nunique(),
      "| 판단번호+항목번호:", b[["decision_no", "item_seq"]].drop_duplicates().shape[0],
      "| 4열 조합:", b[kcols].drop_duplicates().shape[0])
assert b[kcols].drop_duplicates().shape[0] == len(b)
sp = b["bid_notice_no"].fillna("").astype(str).str.split("-", n=1, expand=True)
b["notice_no_base"] = sp[0].replace({"": None}); b["notice_seq"] = sp[1]
b["opening_at"] = pd.to_datetime(b["opening_datetime"], format="%Y-%m-%d %H:%M", errors="coerce")
print("개찰일시 파싱 실패:", int(b["opening_at"].isna().sum()), "| 범위:", str(b["opening_at"].min()), "~", str(b["opening_at"].max()))
b["opening_date"] = b["opening_at"].dt.date
b["opening_ym"] = b["opening_at"].dt.strftime("%Y-%m")
b["is_partial_year"] = 1
b["bid_result_std"] = b["bid_result"].where(b["bid_result"].isin(["낙찰", "유찰"]), "미확인")
b["is_awarded"] = (b["bid_result_std"] == "낙찰").astype(int)
b["budget_amount_usd"] = to_dec(b["budget_amount_usd"])
plan_dno = set(raw("raw_dapa_overseas_plan")["decision_no"].dropna())
b["plan_link_status"] = np.where(b["decision_no"].isin(plan_dno), "연결", "미연결")
b["raw_row_id"] = b["row_id"]; b["cleaned_at"] = NOW; b["cleaned_by"] = USER
print("월별 건수:", b["opening_ym"].value_counts().sort_index().to_dict())
print("bid_result_std:", b["bid_result_std"].value_counts().to_dict(), "| 원본 값:", rb["bid_result"].value_counts().to_dict())
print("plan_link_status(행):", b["plan_link_status"].value_counts().to_dict(),
      "| 판단번호 기준 연결:", b.loc[b["plan_link_status"] == "연결", "decision_no"].nunique(), "/", b["decision_no"].nunique())
print("budget_amount_usd NULL:", int(b["budget_amount_usd"].isna().sum()), "— 달러 표기. 원화 예산과 합산 금지")

원본 전체: 2494
고유성 — 공고번호: 14 | 판단번호: 97 | 판단번호+항목번호: 1362 | 4열 조합: 2494
개찰일시 파싱 실패: 0 | 범위: 2025-03-27 14:00:00 ~ 2025-09-15 14:00:00


월별 건수: {'2025-03': 1073, '2025-04': 901, '2025-05': 5, '2025-06': 1, '2025-07': 3, '2025-08': 314, '2025-09': 197}
bid_result_std: {'유찰': 2146, '낙찰': 348} | 원본 값: {'유찰': 2146, '낙찰': 348}
plan_link_status(행): {'연결': 2432, '미연결': 62} | 판단번호 기준 연결: 83 / 97
budget_amount_usd NULL: 0 — 달러 표기. 원화 예산과 합산 금지


## 5. 적재 (대상이 비어 있을 때만) + `clean_excluded_row` + `meta_load_log`

P3의 제외 행은 계약정보 테스트 계약 6행(`PLACEHOLDER`, §3)뿐이다(열 밀림·키 충돌 0) → 검산은 계약정보만 `raw 6,333 = clean 6,327 + 6`, 나머지 두 표는 `raw = clean + 0`이다.


In [7]:
if count("clean_dapa_overseas_plan_api") > 0:
    print("clean_dapa_overseas_plan_api 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_overseas_plan_api", raw("raw_dapa_overseas_plan_api"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["procure_demand_no", "item_seq", "raw_row_id", "is_item_seq_missing", "demand_year", "army_name_raw", "army_std",
            "army_code", "function_name", "function_code", "item_kind_name", "item_kind_code", "item_name", "stock_no_raw",
            "nsn", "nsn_format", "is_placeholder_nsn", "fsc4", "fsg2", "is_elec", "kdsis_link_status", "equipment_code",
            "equipment_name", "equipment_name_norm", "equipment_name_std", "is_equipment_missing", "quantity", "unit",
            "budget_amount_num", "unit_price_num", "amount_unverified", "progress_status", "is_contracted", "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_dapa_overseas_plan_api", a, cols)
    log_stage(cur, "dapa_overseas_plan_api", "clean_dapa_overseas_plan_api", "중복 처리 후", "조달요구번호+품목순번 고유", n_ins,
              "제외 0(완전 중복·키 충돌·열 밀림 없음). 품목순번 공란 842행은 빈 문자열 + is_item_seq_missing=1",
              f"{NB_TAG} §1", USER)
    log_stage(cur, "dapa_overseas_plan_api", "clean_dapa_overseas_plan_api", "관련 후보", "NSN 있음(13자 숫자·영숫자)",
              int(a["nsn"].notna().sum()),
              f"자리표시·기타 {int(a['nsn'].isna().sum())}행 제외(행은 남기고 nsn=NULL). 숫자13 {int((a['nsn_format']=='숫자13').sum())} · 영숫자13 {int((a['nsn_format']=='영숫자13').sum())}",
              f"{NB_TAG} §1", USER)
    log_stage(cur, "dapa_overseas_plan_api", "clean_dapa_overseas_plan_api", "관련 후보", "전자·통신 FSG 58·59·60",
              int(a["is_elec"].sum()),
              "13자 nsn 앞 2자리 기준. v3 문서 2,270은 길이 무관 stock_no 앞 2자리 집계라 3행 차이. 금액은 통화 미검증이라 건수만 사용",
              f"{NB_TAG} §1", USER)
    log_stage(cur, "dapa_overseas_plan_api", "clean_dapa_overseas_plan_api", "관련 후보", "KDSIS(clean_kdsis_nsn) 연결",
              int((a["kdsis_link_status"] == "연결").sum()),
              f"미연결 {int((a['kdsis_link_status']=='미연결').sum())} · 대상아님(nsn 없음) {int((a['kdsis_link_status']=='대상아님').sum())}. clean_kdsis_nsn.nsn 은 PK 라 다중 일치 0. NSN 연결은 표준화 사실이지 수요 근거가 아님",
              f"{NB_TAG} §1", USER)
    ac = a["army_std"].value_counts().to_dict()
    log_stage(cur, "dapa_overseas_plan_api", "clean_dapa_overseas_plan_api", "관련 후보", "군별 분포(A-6 판단 근거)",
              int(ac.get("공군", 0)),
              f"공군 {ac.get('공군',0)}행({round(100*ac.get('공군',0)/len(a),1)}%)뿐 — 군별 비교는 치우침을 병기해야 한다. 전체 {ac}",
              f"{NB_TAG} §1", USER)
    conn.commit(); print("clean_dapa_overseas_plan_api 적재:", count("clean_dapa_overseas_plan_api"))

if count("ref_equipment_alias") > 0:
    print("ref_equipment_alias 비어 있지 않음 → 건너뜀")
else:
    cols = ["name_raw", "name_norm", "variant_key", "name_std", "variant_group_size", "row_count", "elec_row_count",
            "code_count", "in_elec_scope", "link_status", "basis", "decided_at"]
    n_ins = insert_df("ref_equipment_alias", al, cols)
    ls = al["link_status"].value_counts().to_dict()
    log_stage(cur, "dapa_overseas_plan_api", "ref_equipment_alias", "관련 후보", "적용장비명 원문 → 표준명(자동 매핑)",
              int(ls.get("후보", 0)),
              f"원문 {n_ins}종 중 표기 변이가 관측된 {ls.get('후보',0)}종만 표준명 후보를 채움(link_status=후보). 나머지 {ls.get('미확인',0)}종은 name_std NULL·미확인. 전자 범위 {int(al['in_elec_scope'].sum())}종",
              f"{NB_TAG} §2", USER)
    conn.commit(); print("ref_equipment_alias 적재:", count("ref_equipment_alias"))

if count("clean_dapa_overseas_contract") > 0:
    print("clean_dapa_overseas_contract 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_overseas_contract", raw("raw_dapa_overseas_contract"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["contract_no", "raw_row_id", "contract_name", "contract_form_name", "contract_method_name", "contract_date",
            "contract_year", "period_start", "period_end", "is_open_ended", "contract_org_dept_name", "vendor_name",
            "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_dapa_overseas_contract", c, cols)
    log_stage(cur, "dapa_overseas_contract", "clean_dapa_overseas_contract", "중복 처리 후", "계약번호 고유", n_ins,
              f"테스트 계약 {int(test_mask.sum())}행(업체명 TEST2) PLACEHOLDER 제외. 담당자명 1열·단일값 5열은 clean 에 두지 않음", f"{NB_TAG} §3", USER)
    log_stage(cur, "dapa_overseas_contract", "clean_dapa_overseas_contract", "관련 후보", "고유 대표업체명(원문)",
              int(c["vendor_name"].nunique()),
              "업체명 원문 기준 고유 수. 업체명으로 국가를 추정하지 않으며 금액 열이 없어 집중도 계산 불가", f"{NB_TAG} §3", USER)
    conn.commit(); print("clean_dapa_overseas_contract 적재:", count("clean_dapa_overseas_contract"))

if count("clean_dapa_overseas_bid_result") > 0:
    print("clean_dapa_overseas_bid_result 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_overseas_bid_result", raw("raw_dapa_overseas_bid_result"), USER)   # 원본 전체 단계(파일 파서 행 수)
    # 09-21 저분산 원본 속성 4열 제외(alter_2026-09-21_drop_low_variance.sql): ordering_agency·contract_method·bid_method·award_method(2,489/2,494 동일) — raw에만 보존
    cols = ["raw_row_id", "bid_notice_no", "notice_no_base", "notice_seq", "decision_no", "item_seq", "bid_item_name",
            "opening_at", "opening_date", "opening_ym",
            "is_partial_year", "bid_result_std", "is_awarded", "budget_amount_usd", "plan_link_status", "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_dapa_overseas_bid_result", b, cols)
    br = b["bid_result_std"].value_counts().to_dict()
    log_stage(cur, "dapa_overseas_bid_result", "clean_dapa_overseas_bid_result", "중복 처리 후",
              "raw 1:1(공고번호+판단번호+항목번호+개찰일시 고유)", n_ins,
              f"제외 0. 업무 식별자 단독 고유성 없음(공고번호 {b['bid_notice_no'].nunique()} · 판단번호 {b['decision_no'].nunique()}) → PK 는 raw_row_id",
              f"{NB_TAG} §4", USER)
    log_stage(cur, "dapa_overseas_bid_result", "clean_dapa_overseas_bid_result", "선택 연도 원본",
              "개찰 2025-03~09 부분연도", n_ins,
              f"전 행 부분연도(is_partial_year=1) — 연간 유찰률로 쓰지 않는다. 결과 분포 {br}", f"{NB_TAG} §4", USER)
    log_stage(cur, "dapa_overseas_bid_result", "clean_dapa_overseas_bid_result", "관련 후보",
              "판단번호가 조달계획 파일판에 있는 행", int((b["plan_link_status"] == "연결").sum()),
              f"판단번호 {b['decision_no'].nunique()}개 중 {b.loc[b['plan_link_status']=='연결','decision_no'].nunique()}개 교집합. 행 단위 조인은 하지 않는다",
              f"{NB_TAG} §4", USER)
    conn.commit(); print("clean_dapa_overseas_bid_result 적재:", count("clean_dapa_overseas_bid_result"))

exdf = pd.DataFrame(excluded, columns=["table_name", "raw_row_id", "reason_code", "note"])
if len(exdf):
    cur.execute("SELECT table_name, raw_row_id FROM clean_excluded_row")
    have = set(cur.fetchall())
    exdf = exdf[~exdf.apply(lambda r: (r["table_name"], r["raw_row_id"]) in have, axis=1)]
    if len(exdf):
        exdf["excluded_by"] = USER
        insert_df("clean_excluded_row", exdf, ["table_name", "raw_row_id", "reason_code", "note", "excluded_by"])
        conn.commit()
print("clean_excluded_row 추가:", len(exdf), "| 현재:", count("clean_excluded_row"))

clean_dapa_overseas_plan_api 비어 있지 않음 → 건너뜀
ref_equipment_alias 비어 있지 않음 → 건너뜀
clean_dapa_overseas_contract 비어 있지 않음 → 건너뜀


clean_dapa_overseas_bid_result 비어 있지 않음 → 건너뜀
clean_excluded_row 추가: 0 | 현재: 17


## 6. `ref_fsg`·`ref_fsc` FSG 60 전자 플래그 — 수정 전/후 영향 (명세 §4-5·§4-6, §7 공지 항목)

`db/alter_2026-09-19_p3_clean.sql` §5에서 적용했다(조건부 UPDATE라 재실행해도 0행).

- 수정 전(2026-09-19 실측): `ref_fsg` 60 = 0, `ref_fsc` `fsc2='60'` 24행 전부 0. 수정 후 전부 1(58: 20/20, 59: 26/26, 60: 24/24).
- `ref_fsc` 60군 24행 중 **폐지(status='C') 13행** — 전자 플래그는 폐지 여부와 별개다.
- **`v_b2_fsg_summary`는 값이 바뀌지 않는다**: B2 국산화개발품목(`raw_dapa_localized_item`)에 FSG 60 행이 0건이라 수정 전·후 모두 `is_electronic_group=1`이 2개 군(58·59) · 3,321행 · 고유부품 1,206이다.
- `v_overseas_plan_api_fsc`는 `is_electronic_group`을 ref 에서 읽지 않고 `LEFT(stock_no,2) IN ('58','59')`로 **뷰 안에 직접 박아 두었다** → FSG 60이 반영되지 않는다. `clean_dapa_overseas_plan_api.is_elec`(58·59·60)과 값이 달라지므로 뷰 수정 여부는 팀 확인 뒤로 미룬다(**잠정**, `schema-design.md` §7).
- `clean_kdsis_nsn.is_electronic_group`도 적재 시점 규칙이 `fsg2 IN (58,59)`인 저장 열이라 FSG 60(317행)이 빠져 있다 — P5 담당 영역이라 여기서 고치지 않고 공지만 한다.


In [8]:
display(q("SELECT fsg_code, name_ko, is_electronic_group, note_ko FROM ref_fsg WHERE fsg_code IN ('58','59','60')"))
display(q("SELECT fsc2, COUNT(*) fsc_n, SUM(is_electronic_group) elec_n, SUM(status='C') closed_n FROM ref_fsc WHERE fsc2 IN ('58','59','60') GROUP BY 1"))
display(q("SELECT is_electronic_group, COUNT(*) fsg_rows, SUM(b2_row_count) b2_rows, SUM(b2_part_count) b2_parts FROM v_b2_fsg_summary GROUP BY 1"))
display(q("SELECT COUNT(*) fsg60_rows_in_b2 FROM v_b2_fsg_summary WHERE fsg_code='60'"))   # 0행 기대
display(q("SELECT is_electronic_group, COUNT(*) n FROM clean_kdsis_nsn WHERE fsg2 IN ('58','59','60') GROUP BY 1"))

,fsg_code,name_ko,is_electronic_group,note_ko
0,58,통신·탐지 및 코히런트 방사 장비,1,NaN
1,59,전기 및 전자 장비 구성품,1,NaN
2,60,광섬유 재료·구성품·조립품 및 부속품,1,[2026-09-19] 전자 군급 플래그 0→1(명세 §4-5).


,fsc2,fsc_n,elec_n,closed_n
0,58,20,20,0
1,59,26,26,0
2,60,24,24,13


,is_electronic_group,fsg_rows,b2_rows,b2_parts
0,0,55,30644,11583
1,1,2,3321,1206


,fsg60_rows_in_b2
0,0


,is_electronic_group,n
0,1,33894


## 7. 적재 후 검증 — 검산 `raw = clean + excluded`, 키 규칙, 연결률

In [9]:
_pairs = [("raw_dapa_overseas_plan_api", "clean_dapa_overseas_plan_api"), ("raw_dapa_overseas_contract", "clean_dapa_overseas_contract"), ("raw_dapa_overseas_bid_result", "clean_dapa_overseas_bid_result")]
# 원본 = 파일(read_raw), 정제·제외 = DB. raw = clean + excluded 검산
chk = pd.DataFrame({"tbl": [r for r, _ in _pairs], "raw_n": [len(raw(r)) for r, _ in _pairs], "clean_n": [count(c) for _, c in _pairs]})
_ex = q("SELECT table_name tbl, COUNT(*) excl_n FROM clean_excluded_row GROUP BY table_name")
chk = chk.merge(_ex, on="tbl", how="left").fillna({"excl_n": 0})
chk["excl_n"] = chk["excl_n"].astype(int); chk["gap"] = chk["raw_n"] - chk["clean_n"] - chk["excl_n"]
display(chk)
display(q("SELECT nsn_format, COUNT(*) n, SUM(is_placeholder_nsn) placeholder, SUM(nsn IS NOT NULL) has_nsn FROM clean_dapa_overseas_plan_api GROUP BY 1"))
display(q("SELECT kdsis_link_status, COUNT(*) rows_, COUNT(DISTINCT nsn) nsn_uniq FROM clean_dapa_overseas_plan_api GROUP BY 1"))
display(q("SELECT demand_year, COUNT(*) n, SUM(is_elec) elec, SUM(is_contracted) contracted FROM clean_dapa_overseas_plan_api GROUP BY 1 ORDER BY 1"))
display(q("SELECT army_std, COUNT(*) n, SUM(is_elec) elec FROM clean_dapa_overseas_plan_api GROUP BY 1 ORDER BY n DESC"))
display(q("SELECT fsg2, COUNT(*) n, COUNT(DISTINCT fsc4) fsc4_n FROM clean_dapa_overseas_plan_api WHERE is_elec=1 GROUP BY 1"))
display(q("SELECT procure_demand_no, item_seq, COUNT(*) n FROM clean_dapa_overseas_plan_api GROUP BY 1,2 HAVING n>1"))   # 0행 기대
display(q("SELECT link_status, COUNT(*) n, SUM(in_elec_scope) elec_scope, SUM(name_std IS NOT NULL) has_std FROM ref_equipment_alias GROUP BY 1"))
display(q("SELECT variant_key, COUNT(*) n, MAX(name_std) name_std FROM ref_equipment_alias WHERE variant_group_size>1 GROUP BY 1 ORDER BY n DESC LIMIT 10"))
display(q("SELECT contract_year, COUNT(*) n, COUNT(DISTINCT vendor_name) vendors, SUM(is_open_ended) open_ended FROM clean_dapa_overseas_contract GROUP BY 1 ORDER BY 1"))
display(q("SELECT bid_result_std, COUNT(*) n, SUM(budget_amount_usd IS NULL) usd_null FROM clean_dapa_overseas_bid_result GROUP BY 1"))
display(q("SELECT opening_ym, COUNT(*) n, SUM(is_awarded) awarded FROM clean_dapa_overseas_bid_result GROUP BY 1 ORDER BY 1"))
display(q("SELECT plan_link_status, COUNT(*) rows_, COUNT(DISTINCT decision_no) dno FROM clean_dapa_overseas_bid_result GROUP BY 1"))
display(q("SELECT MIN(log_id) min_log, MAX(log_id) max_log, COUNT(*) n FROM meta_load_log WHERE table_name IN ('clean_dapa_overseas_plan_api','ref_equipment_alias','clean_dapa_overseas_contract','clean_dapa_overseas_bid_result')"))

,tbl,raw_n,clean_n,excl_n,gap
0,raw_dapa_overseas_plan_api,13615,13615,0,0
1,raw_dapa_overseas_contract,6333,6327,6,0
2,raw_dapa_overseas_bid_result,2494,2494,0,0


,nsn_format,n,placeholder,has_nsn
0,자리표시,347,347,0
1,영숫자13,3266,0,3266
2,숫자13,9970,0,9970
3,기타,32,0,0


,kdsis_link_status,rows_,nsn_uniq
0,연결,616,412
1,미연결,12620,7798
2,대상아님,379,0


,demand_year,n,elec,contracted
0,2016,2791,353,2246
1,2017,3044,546,0
2,2018,1,1,0
3,2020,11,0,8
4,2021,2494,398,2362
5,2022,1651,306,1328
6,2023,924,184,719
7,2024,892,176,841
8,2025,854,154,750
9,2026,953,149,224


,army_std,n,elec
0,육군,7404,993
1,해군,5254,1195
2,공군,717,59
3,해병대,212,20
4,국직,18,0
5,미확인,10,0


...

,contract_year,n,vendors,open_ended
0,2017,703,194,31
1,2018,850,199,62
2,2019,824,189,35
3,2020,813,182,41
4,2021,801,184,40
5,2022,667,176,18
6,2023,499,149,17
7,2024,575,151,91
8,2025,595,139,390


,bid_result_std,n,usd_null
0,낙찰,348,0
1,유찰,2146,0


,opening_ym,n,awarded
0,2025-03,1073,178
1,2025-04,901,102
2,2025-05,5,2
3,2025-06,1,0
4,2025-07,3,0
5,2025-08,314,61
6,2025-09,197,5


,plan_link_status,rows_,dno
0,연결,2432,83
1,미연결,62,14


,min_log,max_log,n
0,103,157,13


## 남은 일 (이 노트북이 하지 않은 것)

- `ref_equipment_alias.name_std`가 NULL인 원문(표기 변이가 자료에 없는 것)은 **사람 검수 뒤** `link_status='확정'` + `decided_at`로 채운다. 방위사업청 용어사전(무기체계 별칭 358개, 명세 §9)이 들어오면 그것을 근거로 매핑할 수 있다.
- `레이다`→`레이더` 외의 표기 변이 규칙(띄어쓰기 이미 처리, 괄호 파생형 분리 등)은 추가하지 않았다. 괄호 안 파생형을 떼서 묶는 것은 의미가 달라질 수 있어 **하지 않았다**.
- `v_overseas_plan_api_fsc`의 전자 판정이 뷰 안에 `58·59`로 박혀 있어 FSG 60이 빠진다. `clean_dapa_overseas_plan_api.is_elec`(58·59·60)과 어긋나므로 뷰를 clean 기준으로 바꿀지 팀 확인 필요.
- `clean_kdsis_nsn.is_electronic_group`(P5)도 `58·59` 기준 저장 열 — FSG 60 반영은 P5 담당.
- 금액 열 통화 검증(원화 혼입 의심)은 미해결. 검증 전까지 `amount_unverified=1`이며 화면에서 금액을 쓰지 않는다.
- `clean_dapa_overseas_bid_result` ↔ `clean_dapa_overseas_plan`(파일판) 행 단위 조인은 하지 않았다(`plan_link_status`로 상태만).
